# 🤗 Transformers 入門：Pipeline

<a href="https://colab.research.google.com/github/tjwei/DataScienceSoftware2026/blob/main/06-transformers/01-Pipelines.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

建議環境：Google Colab，`執行階段 → 變更執行階段類型 → T4 GPU`

In [ ]:
!pip install -q -U transformers
# 下載本章練習的參考解答 (q_*.py)
import os, urllib.request
BASE_URL = 'https://raw.githubusercontent.com/tjwei/DataScienceSoftware2026/main/06-transformers/'
for fn in ['q_pipeline.py']:
    if not os.path.exists(fn):
        urllib.request.urlretrieve(BASE_URL + fn, fn)

[Hugging Face Hub](https://huggingface.co/models) 上有超過兩百萬個模型。`transformers` 套件讓我們用幾行程式碼就能使用這些模型。

`pipeline` 是最簡單的用法：把「前處理 → 模型 → 後處理」包在一起。

In [ ]:
import torch
import transformers
from transformers import pipeline
from PIL import Image
from IPython.display import display
from urllib.request import urlopen
print(transformers.__version__)
device = "cuda" if torch.cuda.is_available() else "cpu"

## 情緒分析 (Sentiment Analysis)

In [ ]:
classifier = pipeline("text-classification", model="tabularisai/multilingual-sentiment-analysis", device=device)
classifier(["這家餐廳的牛肉麵超好吃，下次還要再來！",
            "等了一個小時才上菜，服務生態度又很差。",
            "I love this course!"])

## Zero-shot 分類

不需要訓練，直接給候選標籤。（原理是 Natural Language Inference：判斷「這段文字」是否蘊含「這是關於 X 的文章」）

In [ ]:
zero_shot = pipeline("zero-shot-classification", model="MoritzLaurer/mDeBERTa-v3-base-mnli-xnli", device=device)
zero_shot("台積電今天宣布在高雄加蓋兩座 2 奈米晶圓廠",
          candidate_labels=["政治", "財經", "體育", "科技", "娛樂"],
          hypothesis_template="這是一則關於{}的新聞")

## 命名實體辨識 (NER)

中研院 CKIP Lab 訓練的繁體中文模型。

In [ ]:
ner = pipeline("token-classification", model="ckiplab/bert-base-chinese-ner",
               tokenizer="google-bert/bert-base-chinese", aggregation_strategy="simple", device=device)
entities = ner("黃仁勳昨天在台北國際電腦展宣布，NVIDIA 將在北投士林科技園區設立台灣總部。")
entities[:4]

CKIP 的模型使用 BIOES 標記（B=開始、I=中間、E=結尾、S=單字），pipeline 的合併只認得 B/I，所以有些實體會被切開。我們自己把相鄰、同類別的片段合併起來：

In [ ]:
def merge_entities(ents):
    merged = []
    for e in ents:
        if merged and merged[-1]["entity_group"] == e["entity_group"] and merged[-1]["end"] == e["start"]:
            merged[-1]["word"] += e["word"]
            merged[-1]["end"] = e["end"]
        else:
            merged.append(dict(e))
    return [(e["word"].replace(" ", ""), e["entity_group"]) for e in merged]
merge_entities(entities)

## 填空 (Fill-Mask)

BERT 這類「encoder」模型的預訓練方式：把句子中的字遮住，讓模型去猜。

In [ ]:
fill = pipeline("fill-mask", model="google-bert/bert-base-chinese", device=device)
fill(f"台灣最高的山是玉{fill.tokenizer.mask_token}。")

## 文字生成 (Text Generation)：小型 LLM

[Qwen3](https://huggingface.co/Qwen/Qwen3-0.6B) 是阿里巴巴開源的 LLM 系列（Apache 2.0 授權）。0.6B 版本很小，T4 跑起來很快。

給 pipeline 一個「對話 (messages)」列表，它會自動套用 chat template。

In [ ]:
generator = pipeline("text-generation", model="Qwen/Qwen3-0.6B", dtype=torch.float16, device=device)
messages = [{"role": "user", "content": "用一句話解釋什麼是機器學習。/no_think"}]
out = generator(messages, max_new_tokens=128)
answer = out[0]["generated_text"][-1]["content"]
print(answer)
print("去掉空的思考區塊:", answer.split("</think>")[-1].strip())

`/no_think` 是 Qwen3 的指令，關閉「思考模式」。下一個單元會詳細介紹。

## 語音辨識 (Automatic Speech Recognition)

OpenAI Whisper，支援 99 種語言（包含中文）。

In [ ]:
import librosa
audio_url = "https://huggingface.co/datasets/Narsil/asr_dummy/resolve/main/mlk.flac"
with open("mlk.flac", "wb") as f:
    f.write(urlopen(audio_url).read())
audio, sr = librosa.load("mlk.flac", sr=16000)
from IPython.display import Audio
Audio(audio, rate=sr)

In [ ]:
asr = pipeline("automatic-speech-recognition", model="openai/whisper-large-v3-turbo", dtype=torch.float16, device=device)
asr({"raw": audio, "sampling_rate": sr})

## 影像：Zero-shot 影像分類

[SigLIP 2](https://huggingface.co/google/siglip2-base-patch16-224) 同時學了影像和文字的表示，所以可以用任意文字當作類別。

In [ ]:
url = "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/pipeline-cat-chonk.jpeg"
img = Image.open(urlopen(url)).convert("RGB")
display(img.resize((300, int(300 * img.height / img.width))))

In [ ]:
clip = pipeline("zero-shot-image-classification", model="google/siglip2-base-patch16-224", device=device)
clip(img, candidate_labels=["a cat", "a dog", "a lion", "a cat sitting on the grass", "a cat sitting on a sofa"])

## 影像：深度估計 (Depth Estimation)

In [ ]:
depth = pipeline("depth-estimation", model="depth-anything/Depth-Anything-V2-Small-hf", device=device)
result = depth(img)
display(result["depth"].resize((300, int(300 * img.height / img.width))))

## Q
到 [Hugging Face Hub](https://huggingface.co/models) 找一個你有興趣的任務（左側 Tasks 篩選），用 pipeline 試試看。例如：
* `image-classification`
* `object-detection`（例如 `PekingU/rtdetr_v2_r18vd`）
* `image-segmentation`
* `summarization`

In [ ]:
# %load q_pipeline.py
%run -i q_pipeline.py

## 釋放 GPU 記憶體

T4 有 15GB 記憶體，載入太多模型會不夠用。

In [ ]:
import gc
del classifier, zero_shot, ner, fill, generator, asr, clip, depth
gc.collect()
torch.cuda.empty_cache()
print(f"{torch.cuda.memory_allocated() / 1e9:.2f} GB")